In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
import sys
from torch.utils.data import Dataset, DataLoader, RandomSampler
import math
from collections import OrderedDict

In [2]:
!pip install torchinfo
from torchinfo import summary

In [227]:
!pip install sacrebleu bert-score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [228]:
!pip install evaluate
import evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
torch.__version__

'2.2.2'

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len, device):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0., max_len,
                                device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0., d_model, 2, device=device) * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos)
        pe[:, 1::2] = torch.cos(pe_pos)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        out = self.pe[:, :x.size(1)].requires_grad_(False)
        return out

In [6]:
#del pe
pe_e = PositionalEncoding(d_model=768, dropout=0.1, max_len=512, device=device)
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [7]:
pe_e_e = pe_e(inp_tok)
pe_e_e, pe_e_e.shape

(tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
            0.0000e+00,  1.0000e+00],
          [ 8.4147e-01,  5.4030e-01,  8.2843e-01,  ...,  1.0000e+00,
            1.0243e-04,  1.0000e+00],
          [ 9.0930e-01, -4.1615e-01,  9.2799e-01,  ...,  1.0000e+00,
            2.0486e-04,  1.0000e+00],
          ...,
          [-7.5680e-01, -6.5364e-01, -6.9153e-01,  ...,  1.0000e+00,
            4.0971e-04,  1.0000e+00],
          [-9.5892e-01,  2.8366e-01, -9.8573e-01,  ...,  1.0000e+00,
            5.1214e-04,  1.0000e+00],
          [-2.7942e-01,  9.6017e-01, -4.1267e-01,  ...,  1.0000e+00,
            6.1457e-04,  1.0000e+00]]], device='cuda:0'),
 torch.Size([1, 7, 768]))

In [8]:
class Embed(nn.Module):
    def __init__(self, vocab_size, embed_dim, ctx_len, do, device):
        super().__init__()
        self.tok_layer = nn.Embedding(vocab_size, embed_dim)
        self.pos_layer = PositionalEncoding(embed_dim, do, ctx_len, device)
        self.norm_do = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(do)
            )
        self.device = device
        self.ctx_len = ctx_len

    def forward(self, inp):
        inp = inp.to(self.device)
        len_inp = inp.shape[-1]
        try:
            assert len_inp <= self.ctx_len
        except:
            print("Err..Errr. Error...Bro..Length of supplied text exceeds context length defined.Exiting the program now")
            sys.exit(1)
        tok_embed = self.tok_layer(inp)
        pos_embed = self.pos_layer(inp)
        embed_tok_pos = tok_embed + pos_embed
        embed_out = self.norm_do(embed_tok_pos)
        return embed_out

In [9]:
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [10]:
emb = Embed(100, 768, 512, 0.1, device).to(device)

In [11]:
emb_out = emb(inp_tok.to(device))
emb_out.shape, emb_out

(torch.Size([1, 7, 768]),
 tensor([[[-1.5386,  0.3591,  0.8827,  ...,  0.9390,  0.0000,  0.9367],
          [ 1.6824,  0.0000,  1.0312,  ...,  0.0288, -0.0555, -0.2307],
          [-0.0114, -0.0000,  0.9309,  ...,  1.3326, -0.1746,  1.3423],
          ...,
          [-1.6158, -0.7835, -0.6554,  ...,  1.3534, -0.1079,  1.3628],
          [ 0.0432,  1.1828, -0.5430,  ..., -0.9626, -0.7858,  0.6681],
          [ 0.6340, -0.1091, -1.0273,  ...,  0.1483, -2.8837,  1.1835]]],
        device='cuda:0', grad_fn=<NativeDropoutBackward0>))

In [12]:
summary(emb)

Layer (type:depth-idx)                   Param #
Embed                                    --
├─Embedding: 1-1                         76,800
├─PositionalEncoding: 1-2                --
│    └─Dropout: 2-1                      --
├─Sequential: 1-3                        --
│    └─LayerNorm: 2-2                    1,536
│    └─Dropout: 2-3                      --
Total params: 78,336
Trainable params: 78,336
Non-trainable params: 0

In [13]:
def att_mask(attention_mask, lookahead, cross_att, x=None):
    if cross_att:
        batch_dim = x[0]
        repeat = x[1]
    else:
        batch_dim = attention_mask.shape[0]
        repeat = len(attention_mask[0])

    mask =[]
    for i in range(batch_dim):
        am_interim = [attention_mask[i].tolist()] * repeat
        am_interim = torch.tensor(am_interim).unsqueeze(0)
        mask.append(am_interim)
    mask = torch.vstack(mask)
    if lookahead:
        inp_save = mask
        mask = torch.tril(torch.ones(mask.shape))
    mask = torch.where(mask == 0, -torch.inf, 0.0)
    return mask

In [14]:
attention_mask = torch.tensor([[1,1,1,1,0,0], [1,1,1,0,0,0]])
attention_mask.shape

torch.Size([2, 6])

In [15]:
mask  = att_mask(attention_mask, lookahead=False, cross_att=False, x=[2,9,6])

In [16]:
mask, mask.shape

(tensor([[[0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf]],
 
         [[0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf]]]),
 torch.Size([2, 6, 6]))

In [17]:
mask  = att_mask(attention_mask, lookahead=True, cross_att=False, x=[2,9,6])

In [18]:
mask, mask.shape

(tensor([[[0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., -inf, -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., 0., -inf],
          [0., 0., 0., 0., 0., 0.]],
 
         [[0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., -inf, -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., 0., -inf],
          [0., 0., 0., 0., 0., 0.]]]),
 torch.Size([2, 6, 6]))

In [19]:
mask  = att_mask(attention_mask, lookahead=False, cross_att=True, x=[2,9,6])

In [20]:
mask, mask.shape

(tensor([[[0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf]],
 
         [[0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf]]]),
 torch.Size([2, 9, 6]))

In [21]:
attention_mask = torch.tensor([[1,1,1,1,1,1,0,0,0], [1,1,1,1,0,0,0,0,0]])
attention_mask.shape

torch.Size([2, 9])

In [22]:
mask  = att_mask(attention_mask, lookahead=False, cross_att=True, x=[2,6,9])
mask, mask.shape

(tensor([[[0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., 0., 0., 0., -inf, -inf, -inf]],
 
         [[0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
          [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf]]]),
 torch.Size([2, 6, 9]))

In [23]:
class Attention(nn.Module):
    def __init__(self, embed_dim, k_dim, do, device):
        super().__init__()
        self.embed_dim = embed_dim
        self.k_dim = k_dim
        self.query = nn.Linear(embed_dim, k_dim)
        self.key = nn.Linear(embed_dim, k_dim)
        self.value = nn.Linear(embed_dim, k_dim)
        self.att_do = nn.Dropout(do)
        self.device = device

    def forward(self, qry, ky, vlu, mask):
        q = self.query(qry)
        k = self.key(ky)
        v = self.value(vlu)
        qk = (q@k.transpose(1, 2))/(self.k_dim**0.5)
        mask = mask.to(self.device)
        qk_m = qk + mask
        qk_m_smax = torch.softmax(qk_m, dim=-1)
        qk_m_smax_do = self.att_do(qk_m_smax)
        qkv = qk_m_smax_do@v
        return qkv

In [24]:
class Attention_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.num_heads = num_heads
        self.heads_list = [Attention(embed_dim, k_dim, do, device) for i in range(num_heads)]
        self.heads = nn.ModuleList(self.heads_list)
        self.lin_do = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(do)
        )

    def forward(self, q, k, v, mask):
        heads_list_out = [head(q, k, v, mask) for head in self.heads]
        att_head = torch.cat(heads_list_out, dim=-1)
        att_head_out = self.lin_do(att_head)
        return att_head_out

In [25]:
class Encoder_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.MHA = Attention_Block(num_heads, embed_dim, k_dim, do, device)
        self.mha_blk_end_layernorm = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim),
            nn.Dropout(do)
            )
        self.enc_blk_end_layernorm = nn.LayerNorm(embed_dim)

    def forward(self, args_list):  #q, k, v, mask):
        x = args_list[0]
        mask = args_list[1]

        inp_start_att_block = x
        x = self.MHA(x, x, x, mask)

        x = x + inp_start_att_block
        x = self.mha_blk_end_layernorm(x)

        inp_start_ff_block = x
        x = self.ff(x)

        x = x + inp_start_ff_block
        x = self.enc_blk_end_layernorm(x)
        return [x, mask]

In [26]:
class Encoder(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device):
        super().__init__()
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Encoder_Block(num_heads, embed_dim, k_dim, do, device) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)

    def forward(self, input_ids, attention_mask):
        x = self.emb(input_ids)
        mask = att_mask(attention_mask, lookahead=False, cross_att=False, x=None)
        x = self.layers([x, mask])
        return x[0]

In [27]:
enc = Encoder(3, 6, 100, 768, 128, 100, 0.1, device).to(device)
enc

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(100, 768)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-5): 6 x Attention(
            (query): Linear(in_features=768, out_features=128, bias=True)
            (key): Linear(in_features=768, out_features=128, bias=True)
            (value): Linear(in_features=768, out_features=128, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
  

In [28]:
inp = torch.randint(1,100, (4,6))
am = torch.ones(inp.shape)
inp.shape, am.shape

(torch.Size([4, 6]), torch.Size([4, 6]))

In [29]:
out = enc(inp.to(device), am.to(device))
out.shape

torch.Size([4, 6, 768])

In [30]:
class Decoder_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device, decoderonly=False):
        super().__init__()
        self.MHA_CAUSAL_ATTN = Attention_Block(num_heads, embed_dim, k_dim, do, device)
        self.mha_causal_end_layernorm = nn.LayerNorm(embed_dim)
        self.decoderonly = decoderonly
        if not decoderonly:
            self.MHA_CROSS_ATTN = Attention_Block(num_heads, embed_dim, k_dim, do, device)
            self.mha_cross_end_layernorm = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim),
            nn.Dropout(do)
            )
        self.dec_blk_end_layernorm = nn.LayerNorm(embed_dim)

    def forward(self, args_list):
        q = args_list[0]
        enc_k = args_list[1] 
        enc_v = args_list[2] 
        causal_mask = args_list[3] 
        padding_mask = args_list[4]
        
        # Causal Attention Block
        #################################################
        inp_start_causal_att_block = q
        x = self.MHA_CAUSAL_ATTN(q, q, q, causal_mask)
        x = x + inp_start_causal_att_block
        x = self.mha_causal_end_layernorm(x)

        # Cross Attention Block
        ###################################################
        if not self.decoderonly:
            inp_end_causal_att_block = x
            x = self.MHA_CROSS_ATTN(x, enc_k, enc_v, padding_mask)
            x = x + inp_end_causal_att_block
            x = self.mha_cross_end_layernorm(x)
        ###################################################

        inp_start_ff_block = x
        x = self.ff(x)
        x = x + inp_start_ff_block
        x = self.dec_blk_end_layernorm(x)
        return [x, enc_k, enc_v, causal_mask, padding_mask]

In [31]:
class Decoder(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size, embed_dim, 
                 k_dim, ctx_len, do, device):
        super().__init__()
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Decoder_Block(num_heads, embed_dim, k_dim, do, device) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)


    def forward(self, input_ids, attention_mask, enc_attention_mask, enc_k, enc_v):
        x = self.emb(input_ids)
        q = x
        dim = [x.shape[0], attention_mask.shape[1], enc_attention_mask.shape[1]]
        causal_mask = att_mask(attention_mask, lookahead=True, cross_att=False)
        padding_mask = att_mask(enc_attention_mask, lookahead=False, cross_att=True, x=dim)
        x = self.layers([q, enc_k, enc_v, causal_mask, padding_mask])
        return x[0]

In [32]:
inp = torch.randint(1,100, (3,6))
am = torch.ones(inp.shape)
inp.shape, am.shape

(torch.Size([3, 6]), torch.Size([3, 6]))

In [33]:
dec = Decoder(3, 6, 100, 768, 128, 100, 0.1, device).to(device)
dec

Decoder(
  (emb): Embed(
    (tok_layer): Embedding(100, 768)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Decoder_Block(
      (MHA_CAUSAL_ATTN): Attention_Block(
        (heads): ModuleList(
          (0-5): 6 x Attention(
            (query): Linear(in_features=768, out_features=128, bias=True)
            (key): Linear(in_features=768, out_features=128, bias=True)
            (value): Linear(in_features=768, out_features=128, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_causal_end_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (MHA_

In [34]:
inp_e = torch.randint(1,100, (3,7))
am_e = torch.ones(inp_e.shape)
inp_e.shape, am_e.shape, inp_e

(torch.Size([3, 7]),
 torch.Size([3, 7]),
 tensor([[84, 21, 23, 41, 87, 22, 89],
         [28, 60,  8, 19, 95, 18, 16],
         [97, 37, 21, 80, 85, 92, 26]]))

In [35]:
out_e = enc(inp_e.to(device), am_e.to(device))
out_e, out_e.shape

(tensor([[[-1.1520,  0.2251,  0.5948,  ...,  0.4930,  0.7644,  0.3571],
          [ 0.7496, -0.9165,  1.1198,  ...,  1.5222, -0.8834, -0.6800],
          [ 0.5627, -1.3676,  0.5731,  ..., -1.3979,  0.5178, -0.3652],
          ...,
          [-1.1889, -1.6088,  1.2920,  ...,  1.4004, -0.5312,  1.9008],
          [-0.9177, -0.5240, -0.2517,  ..., -0.3394, -0.4634,  1.1644],
          [-1.7263, -0.6355,  0.0976,  ...,  0.4594,  0.2327,  0.0158]],
 
         [[-1.0290,  0.2047,  0.1248,  ...,  1.7964,  0.2256, -0.4307],
          [-0.2000,  0.8311, -0.5451,  ..., -0.2154, -0.7678, -0.0457],
          [ 1.6650, -1.5256, -0.6958,  ...,  0.5390, -1.2662, -0.2836],
          ...,
          [-1.0518, -0.0831, -0.4581,  ...,  1.4901, -0.4471,  1.1940],
          [-1.8780, -0.3381, -2.4201,  ...,  0.4150, -1.5673,  0.8025],
          [-0.7691, -0.6502,  0.3222,  ...,  1.1360, -0.1747, -0.3013]],
 
         [[-0.1468,  0.6034, -0.6218,  ..., -0.9832, -0.5701,  1.5235],
          [-0.7410,  0.0383,

In [36]:
out = dec(inp.to(device), am.to(device), am_e.to(device), out_e, out_e)
out.shape

torch.Size([3, 6, 768])

In [37]:
class Bro_Transformer(nn.Module):

    def __init__(self, num_layers, embed_dim, num_heads,
                 input_vocab_size, target_vocab_size, enc_ctx_len,
                 dec_ctx_len, device, do=0.1):
        super().__init__()
        k_dim = int(embed_dim/num_heads)

        self.encoder = Encoder(num_layers, num_heads, input_vocab_size, embed_dim, k_dim, enc_ctx_len, do, device)
        self.decoder = Decoder(num_layers, num_heads, target_vocab_size, embed_dim, k_dim, dec_ctx_len, do, device)
        self.final_layer = nn.Linear(embed_dim, target_vocab_size)

    def forward(self, enc_input_ids, dec_input_ids, enc_attention_mask, dec_attention_mask):

        enc_output = self.encoder(enc_input_ids, enc_attention_mask)
        dec_output = self.decoder(dec_input_ids, dec_attention_mask,
                                  enc_attention_mask, enc_output, enc_output)
        final_output = self.final_layer(dec_output)
        return final_output

In [38]:
inp_e = torch.randint(1,100, (2,6))
am_e = torch.ones(inp_e.shape)
inp_e.shape, am_e.shape

(torch.Size([2, 6]), torch.Size([2, 6]))

In [39]:
inp_d = torch.randint(1,100, (2,8))
am_d = torch.ones(inp_d.shape)
inp_d.shape, am_d.shape

(torch.Size([2, 8]), torch.Size([2, 8]))

In [40]:
bro_transformer = Bro_Transformer(3, 768, 6, 150, 160, 128, 128, device, 0.1).to(device)

In [41]:
bro_transformer

Bro_Transformer(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(150, 768)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-5): 6 x Attention(
              (query): Linear(in_features=768, out_features=128, bias=True)
              (key): Linear(in_features=768, out_features=128, bias=True)
              (value): Linear(in_features=768, out_features=128, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=768, out_features=768, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernor

In [42]:
out = bro_transformer(inp_e.to(device), inp_d.to(device), am_e.to(device), am_d.to(device))

In [43]:
out.shape

torch.Size([2, 8, 160])

In [44]:
summary(bro_transformer)

Layer (type:depth-idx)                             Param #
Bro_Transformer                                    --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         115,200
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        1,536
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     7,087,872
│    │    └─Encoder_Block: 3-5                     7,087,872
│    │    └─Encoder_Block: 3-6                     7,087,872
├─Decoder: 1-2                                     --
│    └─Embed: 2-3                                  --
│    │    └─Embedding: 3-7                         122,880
│    │    └─PositionalEncoding: 3-8                --
│    │    └─Sequential: 3-9                        1,536
│    └─Sequential: 2-4                             --
│    │    └─Decoder_Block: 3-10         

In [45]:
class BroGPT(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size,
                 embed_dim, ctx_len, do, device):
        super().__init__()
        k_dim = int(embed_dim/num_heads)
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Decoder_Block(num_heads, embed_dim, k_dim, do, device, True) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)
        self.embed_vocab = nn.Linear(embed_dim, vocab_size)

    def forward(self, input_ids, attention_mask): # , enc_k=None, enc_v=None):
        x = self.emb(input_ids)
        causal_mask = att_mask(attention_mask, lookahead=True, cross_att=False)
        padding_mask = att_mask(attention_mask, lookahead=False, cross_att=False)
        #Let us fuse the two together. This is because causal should not be paying attention when there is a padding
        causal_mask = causal_mask + padding_mask
        padding_mask = None
        x = self.layers([x, x, x, causal_mask, padding_mask])
        x = self.embed_vocab(x[0])
        return x

In [46]:
brogpt = BroGPT(num_layers=4, num_heads=6, vocab_size=11000, embed_dim=768,
                ctx_len=512, do=0.1, device=device).to(device)

In [47]:
brogpt

BroGPT(
  (emb): Embed(
    (tok_layer): Embedding(11000, 768)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Decoder_Block(
      (MHA_CAUSAL_ATTN): Attention_Block(
        (heads): ModuleList(
          (0-5): 6 x Attention(
            (query): Linear(in_features=768, out_features=128, bias=True)
            (key): Linear(in_features=768, out_features=128, bias=True)
            (value): Linear(in_features=768, out_features=128, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_causal_end_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ff)

In [48]:
inp_d = torch.randint(1,100, (4,9))
am_d = torch.ones(inp_d.shape)
inp_d.shape, am_d.shape

(torch.Size([4, 9]), torch.Size([4, 9]))

In [49]:
am_d[2]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [50]:
am_d[2] = torch.tensor([1]*6 + [0]*3)
am_d[2], am_d[2].shape

(tensor([1., 1., 1., 1., 1., 1., 0., 0., 0.]), torch.Size([9]))

In [51]:
am_d[3] = torch.tensor([1]*7 + [0]*2)
am_d[3], am_d[3].shape

(tensor([1., 1., 1., 1., 1., 1., 1., 0., 0.]), torch.Size([9]))

In [52]:
am_d

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0.]])

In [53]:
out = brogpt(inp_d.to(device), am_d.to(device))
out.shape

torch.Size([4, 9, 11000])

In [54]:
!pip install transformers datasets sentencepiece sacremoses

In [55]:
from transformers import AutoTokenizer

In [56]:
ckpt = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(ckpt)
tokenizer

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [57]:
%pwd

'/home/ec2-user/SageMaker/MT/brotransformer'

In [58]:
!wget https://gattonweb.uky.edu/faculty/lium/gai/en2fr.zip

--2025-03-15 15:15:39--  https://gattonweb.uky.edu/faculty/lium/gai/en2fr.zip
Resolving gattonweb.uky.edu (gattonweb.uky.edu)... 128.163.2.96
connected. to gattonweb.uky.edu (gattonweb.uky.edu)|128.163.2.96|:443... 
HTTP request sent, awaiting response... 200 OK
Length: 1793553 (1.7M) [application/x-zip-compressed]
Saving to: ‘en2fr.zip.2’

100%[======================================>] 1,793,553   2.10MB/s   in 0.8s   

2025-03-15 15:15:41 (2.10 MB/s) - ‘en2fr.zip.2’ saved [1793553/1793553]



In [59]:
#!unzip en2fr.zip

In [60]:
#!mkdir files
#!mv en2fr.csv ./files/

In [61]:
import pandas as pd

df=pd.read_csv("files/en2fr.csv")
num_examples=len(df)
print(f"there are {num_examples} examples in the training data")
print(df.iloc[30856]["en"])
print(df.iloc[30856]["fr"])

there are 47173 examples in the training data
How are you?
Comment êtes-vous?


In [62]:
tokenized_en=tokenizer.tokenize("I don't speak French.")
print(tokenized_en)
tokenized_fr=tokenizer.tokenize("Je ne parle pas français.")
print(tokenized_fr)
print(tokenizer.tokenize("How are you?"))
print(tokenizer.tokenize("Comment êtes-vous?"))

['I', 'Ġdon', "'t", 'Ġspeak', 'ĠFrench', '.']
['Je', 'Ġne', 'Ġpar', 'le', 'Ġpas', 'Ġfr', 'an', 'Ã§', 'ais', '.']
['How', 'Ġare', 'Ġyou', '?']
['Comment', 'ĠÃ', 'ª', 'tes', '-', 'vous', '?']


In [63]:
tokenizer.get_vocab()

{'Ġdanger': 3514,
 'Ġmentor': 22387,
 'ĠFil': 7066,
 'Ġkilling': 5170,
 'Ġmigrate': 32492,
 'Ãĥ': 5746,
 'ĠAzerbai': 29830,
 'Ġomega': 37615,
 'bringer': 48046,
 'zon': 26361,
 'Ġdefend': 4404,
 'Ġcourtesy': 12537,
 'Adams': 47462,
 'Ġskeletons': 25612,
 'eding': 8228,
 'index': 9630,
 'Ġbegs': 38609,
 'Ġenh': 5881,
 'ĠKiw': 40011,
 'Ġmixer': 33938,
 'Ġenemies': 5775,
 'Ġrescind': 39091,
 'Ġatoms': 23235,
 'Ġsounds': 5238,
 'urn': 700,
 'Ġdelicate': 19217,
 'Ġidiots': 35838,
 'ove': 659,
 'Lewis': 40330,
 'itional': 1859,
 'would': 19188,
 'ĠBeaut': 13711,
 'ĠBefore': 7413,
 'repair': 49932,
 'ĠCSI': 49911,
 'ĠRemote': 21520,
 'ĠWeld': 45156,
 'ĠTraps': 44110,
 'Ġcalmed': 49566,
 'icycle': 35298,
 'Ġlever': 17124,
 'amba': 31842,
 'Ġempty': 6565,
 'want': 42949,
 'Ġdomestically': 42127,
 'ĠHUN': 41041,
 'ĠCompet': 38558,
 'CON': 10943,
 '1990': 19891,
 'ãĤ¨ãĥ«': 46948,
 'Ġlightsaber': 45282,
 'achusetts': 9770,
 'Ġparts': 3354,
 'layout': 39786,
 'Ġsourcing': 47015,
 'Ġoutlines': 27430

In [64]:
df

,Unnamed: 0,en,fr
0,0,"Two young, White males are outside near many b...",Deux jeunes mâles blancs se trouvent à l’extér...
1,0,Several men in hard hats are operating a giant...,Plusieurs hommes portant un chapeau d'assaut f...
2,0,A little girl climbing into a wooden playhouse.,Une petite fille grimpant dans une maison de j...
3,0,A man in a blue shirt is standing on a ladder ...,Un homme en chemise bleue se tient sur une éch...
4,0,Two men are at the stove preparing food.,Deux hommes sont à la cuisinière pour préparer...
...,...,...,...
47168,0,I'm walking with her.,Je marche avec elle.
47169,0,How are you doing today?,Comment allez-vous aujourd'hui?
47170,0,Today is a great day for hiking!,Aujourd'hui est une belle journée pour la rand...
47171,0,What a wonderful day for fishing!,Quelle merveilleuse journée pour la pêche!


In [65]:
en_list = df['en'].tolist()

In [66]:
en_list[:5], len(en_list)

(['Two young, White males are outside near many bushes.',
  'Several men in hard hats are operating a giant pulley system.',
  'A little girl climbing into a wooden playhouse.',
  'A man in a blue shirt is standing on a ladder cleaning a window.',
  'Two men are at the stove preparing food.'],
 47173)

In [67]:
fr_list = df['fr'].tolist()

In [68]:
fr_list[:5], len(fr_list)

(['Deux jeunes mâles blancs se trouvent à l’extérieur près de nombreux buissons.',
  "Plusieurs hommes portant un chapeau d'assaut font fonctionner un système géant de poulies.",
  'Une petite fille grimpant dans une maison de jeux en bois.',
  'Un homme en chemise bleue se tient sur une échelle pour nettoyer une fenêtre.',
  'Deux hommes sont à la cuisinière pour préparer la nourriture.'],
 47173)

In [69]:
en_train_corpus = (en_list[i:i+1000] for i in range(0, len(en_list), 1000))

In [70]:
en_train_corpus

<generator object <genexpr> at 0x7f42d30cc660>

In [71]:
#len((next(en_train_corpus)))

In [72]:
fr_train_corpus = (fr_list[i:i+1000] for i in range(0, len(fr_list), 1000))

In [73]:
fr_train_corpus

<generator object <genexpr> at 0x7f42d30cc2e0>

In [74]:
#len(next(fr_train_corpus))

In [75]:
del tokenizer

In [76]:
orig_tokenizer = AutoTokenizer.from_pretrained(ckpt)
orig_tokenizer

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [77]:
en_corp = ' '.join(en_list)
print(len(en_corp))
unique_en_word_toks = set(en_corp.split())
len(unique_en_word_toks)

2389158


21725

In [78]:
fr_corp = ' '.join(fr_list)
print(len(fr_corp))
unique_fr_word_toks = set(fr_corp.split())
len(unique_fr_word_toks)

2756011


25656

In [79]:
#We want to restrict the tokenizer size to 11500
#we will create two tokenizers - one for english and another for french

In [80]:
#let us reinitialize the generator object so that their counters are reset
en_train_corpus = (en_list[i:i+1000] for i in range(0, len(en_list), 1000))
fr_train_corpus = (fr_list[i:i+1000] for i in range(0, len(fr_list), 1000))

In [81]:
en_train_corpus, fr_train_corpus

(<generator object <genexpr> at 0x7f42d30cf7d0>,
 <generator object <genexpr> at 0x7f42d30cf5a0>)

In [82]:
en_tokenizer = orig_tokenizer.train_new_from_iterator(en_train_corpus, 11500) #vocab size of 11500

In [83]:
en_tokenizer

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=11500, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [84]:
orig_tokenizer

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [85]:
fr_tokenizer = orig_tokenizer.train_new_from_iterator(fr_train_corpus, 11500) #vocab size of 11500

In [86]:
print("###Tokenizing English and French using original tokenizer###")
tokenized_en=orig_tokenizer.tokenize("I don't speak French.")
print(tokenized_en)
tokenized_fr=orig_tokenizer.tokenize("Je ne parle pas français.")
print(tokenized_fr)
print(orig_tokenizer.tokenize("How are you?"))
print(orig_tokenizer.tokenize("Comment êtes-vous?"))

###Tokenizing English and French using original tokenizer###
['I', 'Ġdon', "'t", 'Ġspeak', 'ĠFrench', '.']
['Je', 'Ġne', 'Ġpar', 'le', 'Ġpas', 'Ġfr', 'an', 'Ã§', 'ais', '.']
['How', 'Ġare', 'Ġyou', '?']
['Comment', 'ĠÃ', 'ª', 'tes', '-', 'vous', '?']


In [87]:
print("###Tokenizing English and French using new  tokenizers###")
tokenized_en=en_tokenizer.tokenize("I don't speak French.")
print(tokenized_en)
tokenized_fr=fr_tokenizer.tokenize("Je ne parle pas français.")
print(tokenized_fr)
print(en_tokenizer.tokenize("How are you?"))
print(fr_tokenizer.tokenize("Comment êtes-vous?"))

###Tokenizing English and French using new  tokenizers###
['I', 'Ġdon', "'t", 'Ġspeak', 'ĠFrench', '.']
['Je', 'Ġne', 'Ġparle', 'Ġpas', 'ĠfranÃ§ais', '.']
['How', 'Ġare', 'Ġyou', '?']
['Comment', 'ĠÃªtes', '-', 'vous', '?']


In [88]:
fr_tokenizer.decode(fr_tokenizer.encode("Comment êtes-vous?"))

'Comment êtes-vous?'

In [89]:
fr_tokenizer.decode(fr_tokenizer.encode("Je ne parle pas français."))

'Je ne parle pas français.'

In [90]:
#As we can see new tokenizer is able to tokenize french text better. 
#at the same time we can limit the vocab size which will bring down the model size

In [91]:
#let us try to use en_tokenizer to tokenize french words and see the fun and vice versa

In [92]:
en_tokenizer.tokenize("Je ne parle pas français.") ##oh bro what's this..dont fret. this is expected

['J',
 'e',
 'Ġne',
 'Ġpar',
 'le',
 'Ġp',
 'as',
 'Ġfr',
 'an',
 'Ã',
 '§',
 'a',
 'is',
 '.']

In [93]:
fr_tokenizer.tokenize("I don't speak French.")

['I', 'Ġdon', "'", 't', 'Ġs', 'pe', 'ak', 'ĠFr', 'en', 'ch', '.']

In [94]:
en_tokenizer.tokenize("I don't speak French.")

['I', 'Ġdon', "'t", 'Ġspeak', 'ĠFrench', '.']

In [95]:
#en_tokenizer.save_pretrained('./tokenizers_dir/en_tokenizer')

In [96]:
#fr_tokenizer.save_pretrained('./tokenizers_dir/fr_tokenizer')

In [97]:
#new_en_tokenizer = AutoTokenizer.from_pretrained('./tokenizers_dir/en_tokenizer')
#new_en_tokenizer

In [98]:
#new_fr_tokenizer = AutoTokenizer.from_pretrained('./tokenizers_dir/fr_tokenizer')
#new_fr_tokenizer

In [99]:
#print("###Tokenizing English and French using new  tokenizers initialized from saved config###")
#tokenized_en=new_en_tokenizer.tokenize("I don't speak French.")
#print(tokenized_en)
#tokenized_fr=new_fr_tokenizer.tokenize("Je ne parle pas français.")
#print(tokenized_fr)
#print(new_en_tokenizer.tokenize("How are you?"))
#print(new_fr_tokenizer.tokenize("Comment êtes-vous?"))

In [100]:
#new_en_tokenizer("I don't speak French.")

In [101]:
#new_fr_tokenizer.tokenize("Je ne parle pas français.")

In [102]:
del  fr_tokenizer

In [103]:
en_tokenizer = AutoTokenizer.from_pretrained('./tokenizers_dir/en_tokenizer')
en_tokenizer

GPT2TokenizerFast(name_or_path='./tokenizers_dir/en_tokenizer', vocab_size=11500, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [104]:
fr_tokenizer = AutoTokenizer.from_pretrained('./tokenizers_dir/fr_tokenizer')
fr_tokenizer

GPT2TokenizerFast(name_or_path='./tokenizers_dir/fr_tokenizer', vocab_size=11500, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

What all do we need to train model?

1) Model initialized and moved to GPU
2) Tokenizer initialized
3) Dataset
4) DataLoader
5) Collate_fn
6) loss function
7) optimizer
8) training loop
9) Split dataset into train/validation/test dataset
10) evaluator = loss/accuracy on train and eval dataset
11) Once training is done we need to use Bleu Score and BERT score to assess efficiency

In [105]:
en_vocab_size = en_tokenizer.vocab_size
fr_vocab_size = fr_tokenizer.vocab_size
en_vocab_size, fr_vocab_size

(11500, 11500)

In [106]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [107]:
bro_model = Bro_Transformer(num_layers=6, embed_dim=256, num_heads=8,
                            input_vocab_size=en_vocab_size, target_vocab_size=fr_vocab_size,
                            enc_ctx_len=128, dec_ctx_len=128, device=device, do=0.1)

In [108]:
bro_model

Bro_Transformer(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(11500, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm

In [109]:
summary(bro_model)

Layer (type:depth-idx)                             Param #
Bro_Transformer                                    --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         2,944,000
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        512
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     789,760
│    │    └─Encoder_Block: 3-5                     789,760
│    │    └─Encoder_Block: 3-6                     789,760
│    │    └─Encoder_Block: 3-7                     789,760
│    │    └─Encoder_Block: 3-8                     789,760
│    │    └─Encoder_Block: 3-9                     789,760
├─Decoder: 1-2                                     --
│    └─Embed: 2-3                                  --
│    │    └─Embedding: 3-10                        2,944,000
│    │    └─PositionalEncoding: 

In [110]:
bro_model.to(device)

Bro_Transformer(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(11500, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm

In [111]:
df

,Unnamed: 0,en,fr
0,0,"Two young, White males are outside near many b...",Deux jeunes mâles blancs se trouvent à l’extér...
1,0,Several men in hard hats are operating a giant...,Plusieurs hommes portant un chapeau d'assaut f...
2,0,A little girl climbing into a wooden playhouse.,Une petite fille grimpant dans une maison de j...
3,0,A man in a blue shirt is standing on a ladder ...,Un homme en chemise bleue se tient sur une éch...
4,0,Two men are at the stove preparing food.,Deux hommes sont à la cuisinière pour préparer...
...,...,...,...
47168,0,I'm walking with her.,Je marche avec elle.
47169,0,How are you doing today?,Comment allez-vous aujourd'hui?
47170,0,Today is a great day for hiking!,Aujourd'hui est une belle journée pour la rand...
47171,0,What a wonderful day for fishing!,Quelle merveilleuse journée pour la pêche!


In [112]:
df['en_len'] = df['en'].apply(lambda x: len(en_tokenizer(x).input_ids))

In [113]:
df['fr_len'] = df['fr'].apply(lambda x: len(fr_tokenizer(x).input_ids))

In [114]:
df.describe()

,Unnamed: 0,en_len,fr_len
count,47173.0,47173.000000,47173.000000
mean,0.0,11.248447,13.009370
std,0.0,4.563858,5.564908
min,0.0,2.000000,2.000000
25%,0.0,8.000000,9.000000
50%,0.0,10.000000,12.000000
75%,0.0,14.000000,16.000000
max,0.0,44.000000,54.000000


In [115]:
df['en_len'].quantile(0.99), df['fr_len'].quantile(0.99)

(25.0, 30.0)

In [116]:
df_lte32 = df[(df['en_len'] <= 32) & (df['fr_len'] <= 32)]

In [117]:
len(df), len(df_lte32)

(47173, 46883)

In [118]:
len(df) - len(df_lte32)

290

In [119]:
# we do not want to exceed 512 tokens so taking batch size of 16 and 31 tokens
16*31
## We will get rid of data that exceed 31 tokens

496

In [120]:
df_lt32 = df[(df['en_len'] < 32) & (df['fr_len'] < 32)].copy()

In [121]:
df_lt32.reset_index(inplace=True, drop=True)
df_lt32.describe()

,Unnamed: 0,en_len,fr_len
count,46798.0,46798.000000,46798.000000
mean,0.0,11.118082,12.830976
std,0.0,4.321121,5.204432
min,0.0,2.000000,2.000000
25%,0.0,8.000000,9.000000
50%,0.0,10.000000,12.000000
75%,0.0,14.000000,16.000000
max,0.0,31.000000,31.000000


In [122]:
df_lt32.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46798 entries, 0 to 46797
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  46798 non-null  int64 
 1   en          46798 non-null  object
 2   fr          46798 non-null  object
 3   en_len      46798 non-null  int64 
 4   fr_len      46798 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 1.8+ MB


In [123]:
df_lt32

,Unnamed: 0,en,fr,en_len,fr_len
0,0,"Two young, White males are outside near many b...",Deux jeunes mâles blancs se trouvent à l’extér...,11,15
1,0,Several men in hard hats are operating a giant...,Plusieurs hommes portant un chapeau d'assaut f...,12,18
2,0,A little girl climbing into a wooden playhouse.,Une petite fille grimpant dans une maison de j...,9,12
3,0,A man in a blue shirt is standing on a ladder ...,Un homme en chemise bleue se tient sur une éch...,15,15
4,0,Two men are at the stove preparing food.,Deux hommes sont à la cuisinière pour préparer...,9,11
...,...,...,...,...,...
46793,0,I'm walking with her.,Je marche avec elle.,6,5
46794,0,How are you doing today?,Comment allez-vous aujourd'hui?,6,8
46795,0,Today is a great day for hiking!,Aujourd'hui est une belle journée pour la rand...,8,12
46796,0,What a wonderful day for fishing!,Quelle merveilleuse journée pour la pêche!,7,7


In [135]:
class EN_FR_DS(Dataset):
    def __init__(self, df, en_tokenizer, fr_tokenizer):
        self.data = df
        self.src_tokenizer = en_tokenizer
        self.tgt_tokenizer = fr_tokenizer
        self.EOT = self.tgt_tokenizer.encode('<|endoftext|>')

    def __len__(self):
        return len(df)

    def __getitem__(self, idx):
        en_item = self.data.iloc[idx]['en']
        fr_item = '<|endoftext|>' + self.data.iloc[idx]['fr']
        return en_item, fr_item

In [136]:
ds = EN_FR_DS(df, en_tokenizer, fr_tokenizer)

In [137]:
ds[0], len(ds[0])

(('Two young, White males are outside near many bushes.',
  '<|endoftext|>Deux jeunes mâles blancs se trouvent à l’extérieur près de nombreux buissons.'),
 2)

In [138]:
ds[9], len(ds[9])

(('Boys dancing on poles in the middle of the night.',
  '<|endoftext|>Les garçons dansent sur des mâts au milieu de la nuit.'),
 2)

print(en_tokenizer.decode(ds[0][0]))
print(fr_tokenizer.decode(ds[0][2]))
print(df.iloc[0]['en'])
print(df.iloc[0]['fr'])
print(fr_tokenizer.decode(ds[0][4]))

print(en_tokenizer.decode(ds[1][0]))
print(fr_tokenizer.decode(ds[1][2]))
print(df.iloc[1]['en'])
print(df.iloc[1]['fr'])
print(fr_tokenizer.decode(ds[1][4]))

In [139]:
import os
dl = DataLoader(ds, shuffle=True, batch_size=16, num_workers=os.cpu_count())
dl

In [140]:
dl_iter = iter(dl)
dl_iter

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [141]:
next(dl_iter)

[('He lied to us.',
  "You should follow your teacher's advice.",
  'He does not like being punished.',
  'A youth baseball player rounds third as the outfielders retrieve the ball at the fence.',
  'Mary senses the aliens are watching her.',
  'The ladies in the purple jackets are playing the sax.',
  'I woke up at 2:30.',
  'We go fishing from time to time.',
  'Two older men in coats are standing outside.',
  'A middle-aged man wearing a red hat is fishing.',
  'You or I will be chosen.',
  'Girl in red, with striped tights, playing violin outside of building.',
  'A man with a turquoise vest and black pants sits down in front of a brick monument.',
  'Four dogs jumping over a hurdle.',
  'A blond-hair lady sitting with a kid.',
  'Get away!'),
 ('<|endoftext|>Il nous a menti.',
  '<|endoftext|>Vous devriez suivre les conseils de votre professeur.',
  "<|endoftext|>Il n'aime pas être puni.",
  "<|endoftext|>Un jeune joueur de baseball se classe au troisième rang alors que les joueur

In [142]:
batch = next(dl_iter)

In [143]:
batch[0]

('All of them are not present.',
 'A man and a woman with feathers on her head dance.',
 'A group of Native Americans sit in a circle chanting and beating a drum.',
 'A group of adults and children are at an amusement park watching over a railing.',
 'Young kids playing soccer on a lush, green field.',
 'Bringing up a baby is hard work.',
 'A young boy taking a picture through the glass and two females chatting.',
 'My aunt made me a new skirt.',
 'Tom seems to hardly ever get his homework done on time.',
 'I envied his new house.',
 'They kept me waiting for an hour.',
 'A man wearing a white shirt is skateboarding downhill on a road.',
 "It's up to you.",
 'A young indian mother is with her child.',
 "He took care of the business after his father's death.",
 'I wish you had come with us.')

In [144]:
batch[1]

('<|endoftext|>Tous ne sont pas présents.',
 '<|endoftext|>Un homme et une femme avec des plumes sur la tête dansent.',
 "<|endoftext|>Un groupe d'Américains autochtones s'asseoir dans un cercle chantant et battant un tambour.",
 "<|endoftext|>Un groupe d'adultes et d'enfants se trouvent dans un parc d'attractions et surveillent un rail.",
 '<|endoftext|>Jeunes enfants jouant au soccer sur un terrain luxuriant et verdoyant.',
 '<|endoftext|>Élever un bébé est un travail ardu.',
 '<|endoftext|>Un jeune garçon prenant une photo à travers le verre et deux femelles chattant.',
 "<|endoftext|>Ma tante m'a fait une nouvelle jupe.",
 '<|endoftext|>Il semble que Tom ne fasse guère ses devoirs à temps.',
 "<|endoftext|>J'envieais sa nouvelle maison.",
 "<|endoftext|>Ils m'ont maintenu en attente pendant une heure.",
 '<|endoftext|>Un homme portant une chemise blanche fait du skateboarding sur une route.',
 "<|endoftext|>C'est à vous de choisir.",
 '<|endoftext|>Une jeune mère indienne est avec 

In [145]:
def collate_fn(batch, en_tokenizer, fr_tokenizer):
    eos_tokenid = fr_tokenizer.encode('<|endoftext|>')
    pad_tokenid = fr_tokenizer.encode('<|endoftext|>')
    ignore_loss_id = [-100]  #ignore loss
    ignore_token_id = [0]   #attention mask filler
    x_input_ids, x_am = [], []
    y_input_ids, y_am, lab_list = [], [], []

    for x, y in batch:
        x_tok = en_tokenizer(x)
        y_tok = fr_tokenizer(y)

        x_tok_input_ids = x_tok.input_ids
        x_tok_am = x_tok.attention_mask

        y_tok_input_ids = y_tok.input_ids
        y_tok_am = y_tok.attention_mask

        labs = y_tok_input_ids[1:] + eos_tokenid

        x_input_ids.append(x_tok_input_ids)
        x_am.append(x_tok_am)

        y_input_ids.append(y_tok_input_ids)
        y_am.append(y_tok_am)

        lab_list.append(labs)

    len_enc = [len(l) for l in x_input_ids]
    max_len_enc = max(len_enc)

    len_labs = [len(l) for l in lab_list]
    max_len_labs = max(len_labs)

    max_len = max([max_len_enc, max_len_labs])

    def pad_tokens_stack(in_list, max_len, padding):
        list_tensor = []
        for item in in_list:
            len_item = len(item)
            deficit = max_len - len_item
            if deficit > 0:
                item = item + padding*deficit
            list_tensor.append(torch.tensor(item))
        item_ids = torch.vstack(list_tensor)
        return item_ids

    enc_input_ids = pad_tokens_stack(x_input_ids, max_len, pad_tokenid)
    enc_am = pad_tokens_stack(x_am, max_len, ignore_token_id)

    dec_input_ids = pad_tokens_stack(y_input_ids, max_len, pad_tokenid)
    dec_am = pad_tokens_stack(y_am, max_len, ignore_token_id)

    labels = pad_tokens_stack(lab_list, max_len, ignore_loss_id)

    return {'enc_input_ids': enc_input_ids, 'dec_input_ids': dec_input_ids,
            'enc_attention_mask': enc_am, 'dec_attention_mask': dec_am}, labels

In [146]:
from functools import partial
wrapper_collate_fn = partial(
    collate_fn,
    en_tokenizer=en_tokenizer,
    fr_tokenizer=fr_tokenizer
    )

dl = DataLoader(ds, shuffle=False, batch_size=3,
                num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)

In [147]:
iter_dl = iter(dl)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [148]:
batch = next(iter_dl)

In [149]:
len(batch)

2

In [150]:
batch

({'enc_input_ids': tensor([[ 351,  388,   12, 6932, 2799,  320,  541,  609, 1015, 4038,   14,    0,
              0,    0,    0,    0,    0,    0,    0],
          [ 959,  426,  271, 1033, 1353,  320, 3778,  257, 2669, 9013, 4759,   14,
              0,    0,    0,    0,    0,    0,    0],
          [  33,  520,  390, 1175,  566,  257, 1230, 9739,   14,    0,    0,    0,
              0,    0,    0,    0,    0,    0,    0]]),
  'dec_input_ids': tensor([[   0,  369,  654, 5144, 1118,  329,  764,  292,  265,  611,  513,  559,
            271, 1808, 5170,   14,    0,    0,    0],
          [   0,  958,  451,  370,  267,  569,  257,    7,  985,  613, 1155, 6561,
            267, 5512, 5717,  271, 9658, 1528,   14],
          [   0,  327,  672,  437, 4822,  305,  293,  806,  271, 1960,  291,  698,
             14,    0,    0,    0,    0,    0,    0]]),
  'enc_attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,

In [151]:
print(batch[0]['enc_input_ids'].shape)
print(batch[0]['dec_input_ids'].shape)
print(batch[0]['enc_attention_mask'].shape)
print(batch[0]['dec_attention_mask'].shape)
print(batch[1].shape)

torch.Size([3, 19])
torch.Size([3, 19])
torch.Size([3, 19])
torch.Size([3, 19])
torch.Size([3, 19])


In [152]:
batch = next(iter_dl)

In [153]:
print(batch[0]['enc_input_ids'].shape)
print(batch[0]['dec_input_ids'].shape)
print(batch[0]['enc_attention_mask'].shape)
print(batch[0]['dec_attention_mask'].shape)
print(batch[1].shape)

torch.Size([3, 16])
torch.Size([3, 16])
torch.Size([3, 16])
torch.Size([3, 16])
torch.Size([3, 16])


In [154]:
batch

({'enc_input_ids': tensor([[  33,  298,  271,  257,  405,  378,  290,  433,  289,  257, 2189, 2181,
            257, 1025,   14,    0],
          [ 351,  426,  320,  329,  277, 3607, 1568,  774,   14,    0,    0,    0,
              0,    0,    0,    0],
          [  33,  298,  271,  523,  884,  257,  761,  402,  277,  533,  298, 4937,
            375,  378,   14,    0]]),
  'dec_input_ids': tensor([[   0,  274,  314,  291,  409,  590,  329,  473,  309,  293, 2646,  413,
           3358,  293, 1254,   14],
          [   0,  369,  451,  461,  292,  308, 5701,  413, 3687,  308, 1198,   14,
              0,    0,    0,    0],
          [   0,  274,  314,  745,  473,  293,  844,  624,  377,  265,    7,  769,
           4662,  406,  409,   14]]),
  'enc_attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]]),
  'dec_attention_mask': tensor([[1, 1, 1, 1,

In [155]:
batch = next(iter_dl)

In [156]:
print(batch[0]['enc_input_ids'].shape)
print(batch[0]['dec_input_ids'].shape)
print(batch[0]['enc_attention_mask'].shape)
print(batch[0]['dec_attention_mask'].shape)
print(batch[1].shape)

torch.Size([3, 22])
torch.Size([3, 22])
torch.Size([3, 22])
torch.Size([3, 22])
torch.Size([3, 22])


In [157]:
batch

({'enc_input_ids': tensor([[  33,  298,  290,  845,  329,  257, 3117, 5260,    0,    0,    0,    0,
              0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [  33, 1049, 9643,  390,  773,  289,  408, 1376,  402, 8064, 3784,  441,
            277,  439,   14,    0,    0,    0,    0,    0,    0,    0],
          [  33,  336,  307,  257,  548, 2293,  290,  455,  460,  257, 3129,   14,
              0,    0,    0,    0,    0,    0,    0,    0,    0,    0]]),
  'dec_input_ids': tensor([[   0,  274,  314, 1250,  292,  267, 6063, 2228,    0,    0,    0,    0,
              0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
          [   0,  327,  441,  437,  292,  308, 3842, 1549,  309,  438,  940, 1428,
            674,  291,  263,    7, 8214, 4331,  305,  308,  459,   14],
          [   0,  327,  343, 6433,  257,    7,  258,  642,  741,  329, 1074,  559,
            257,    7,  331,  565,   14,    0,    0,    0,    0,    0]]),
  'enc_attention_mask': tensor([

In [158]:
batch[0].items()

dict_items([('enc_input_ids', tensor([[  33,  298,  290,  845,  329,  257, 3117, 5260,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        [  33, 1049, 9643,  390,  773,  289,  408, 1376,  402, 8064, 3784,  441,
          277,  439,   14,    0,    0,    0,    0,    0,    0,    0],
        [  33,  336,  307,  257,  548, 2293,  290,  455,  460,  257, 3129,   14,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0]])), ('dec_input_ids', tensor([[   0,  274,  314, 1250,  292,  267, 6063, 2228,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        [   0,  327,  441,  437,  292,  308, 3842, 1549,  309,  438,  940, 1428,
          674,  291,  263,    7, 8214, 4331,  305,  308,  459,   14],
        [   0,  327,  343, 6433,  257,    7,  258,  642,  741,  329, 1074,  559,
          257,    7,  331,  565,   14,    0,    0,    0,    0,    0]])), ('enc_attention_mask', tensor([[1, 1, 1,

In [159]:
batch[0].keys()

dict_keys(['enc_input_ids', 'dec_input_ids', 'enc_attention_mask', 'dec_attention_mask'])

In [160]:
from torch.optim import AdamW

In [161]:
def train_model(num_epochs, lrate, brogpt, dl, device, vocab_size):
    epochs = num_epochs
    lr = lrate
    fn_loss = nn.CrossEntropyLoss()
    optimizer = AdamW(brogpt.parameters(), lr=lr)
    grad_accum_steps = 1 #args.grad_accum_steps # 4 - previous static value

    for i in range(epochs):
        loss_epoch = 0
        n_step = 0
        grad_accum_counter = 1

        for inputs in dl:
            data = inputs[0]
            label = inputs[1]
            batch_size = label.shape[0]
            ctx_size = label.shape[1]
            data = {i: k.to(device) for i, k in data.items()}
            label = label.to(device)
            #enc_input_ids, dec_input_ids, enc_attention_mask, dec_attention_mask)
            #dict_keys(['enc_input_ids', 'enc_am', 'dec_input_ids', 'dec_am'])
            out = brogpt(**data)
            #out = out.logits  # Code added for hugging face based transformer models
            out = out.view(batch_size * ctx_size, vocab_size)
            label = label.view(batch_size * ctx_size)
            loss = fn_loss(out, label)
            loss = loss / grad_accum_steps
            loss.backward()
            if grad_accum_counter == grad_accum_steps:
                #logger.info(f'Steps: {grad_accum_counter}, adjusting learnable params now')
                optimizer.step()
                optimizer.zero_grad()
                grad_accum_counter = 0
            loss_epoch = loss_epoch + (loss.item() * batch_size * grad_accum_steps)
            #if n_step % 100 == 0:
            #    print(f'Step: {n_step}, Loss: {loss.item()}')
            grad_accum_counter += 1
            #n_step += 1
            ##update so that if remaining dataloader runs are less than grad accum steps then at the last run i should
            #gradient update
        average_loss = loss_epoch/len(dl.dataset)
        perplexity = math.exp(average_loss) 
        print(f'Epoch: {i} -- Average loss: {average_loss}')
        print(f'Epoch: {i} -- Perplexity: {perplexity}')

    return brogpt, optimizer, average_loss, perplexity


In [162]:
dl = DataLoader(ds, shuffle=False, batch_size=256,
                num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)

In [163]:
vocab_size = fr_tokenizer.vocab_size
vocab_size

11500

In [167]:
model_dict = torch.load('./MT_bro/ckpt_Epoch_100_Prplxty_1.0323989382902838.pt')

In [168]:
model_dict.keys()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss', 'device'])

In [169]:
bro_model.load_state_dict(model_dict['model_state_dict'])

<All keys matched successfully>

In [170]:
torch.cuda.empty_cache()
bro_model, optimizer, average_loss, perplexity = train_model(40, 0.0002, bro_model, dl, device, vocab_size)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch: 0 -- Average loss: 0.1937053845523891
Epoch: 0 -- Perplexity: 1.2137386441219367


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
TOKENIZERS_PARALLELISM=(true | false)iable 
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
TOKENIZERS_PARALLELISM=(true | false)iable 
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. 

Epoch: 1 -- Average loss: 0.08478394283215389
Epoch: 1 -- Perplexity: 1.0884818669815894


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

...

Epoch: 37 -- Average loss: 0.015134095062999224
Epoch: 37 -- Perplexity: 1.0152491953930987


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch: 38 -- Average loss: 0.015393329683882451
Epoch: 38 -- Perplexity: 1.0155124172500034


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch: 39 -- Average loss: 0.015740576204905713
Epoch: 39 -- Perplexity: 1.0158651116363546


In [171]:
torch.cuda.empty_cache()

In [172]:
ckpt_name = './MT_bro/' + 'ckpt_' + "Epoch_" + str(140) + "_Prplxty_" + str(perplexity) + '.pt'
ckpt_name

'./MT_bro/ckpt_Epoch_140_Prplxty_1.0158651116363546.pt'

In [173]:
os.makedirs('./MT_bro/', exist_ok=True)

In [174]:
torch.save({
            'epoch': 140,
            'model_state_dict': bro_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': average_loss,
            'device': device
            }, ckpt_name)

In [175]:
!ls -ltrh {ckpt_name}

-rw-rw-r-- 1 ec2-user ec2-user 230M Mar 15 16:43 ./MT_bro/ckpt_Epoch_140_Prplxty_1.0158651116363546.pt


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [176]:
fr_tokenizer.bos_token

'<|endoftext|>'

In [177]:
fr_tokenizer.vocab[fr_tokenizer.bos_token]

0

In [178]:
fr_tokenizer("Un")

{'input_ids': [274], 'attention_mask': [1]}

In [229]:
bleu_metric = evaluate.load('sacrebleu')
bert_metric = evaluate.load('bertscore')

In [213]:
def translate(en_text, fr_text, fr_start, en_tokenizer, fr_tokenizer, bro_model, compute_metrics=False):
    bro_model.eval()
    tokenized_en = en_tokenizer(en_text)
    tokenized_fr = fr_tokenizer(fr_start)
    enc_input_ids = torch.tensor(tokenized_en.input_ids).unsqueeze(0).to(device)
    dec_input_ids = torch.tensor(tokenized_fr.input_ids).unsqueeze(0).to(device)
    enc_attention_mask = torch.tensor(tokenized_en.attention_mask).unsqueeze(0).to(device)
    dec_attention_mask = torch.tensor(tokenized_fr.attention_mask).unsqueeze(0).to(device)
    pred = 12345678
    while pred!=0:
        with torch.no_grad():
            output = bro_model(enc_input_ids=enc_input_ids, dec_input_ids=dec_input_ids,
                               enc_attention_mask=enc_attention_mask, dec_attention_mask=dec_attention_mask)
        prediction = torch.argmax(output, dim=-1)[0][-1].unsqueeze(0).unsqueeze(0)
        #print(prediction)
        pred = torch.argmax(output, dim=-1).tolist()[0][-1]
        dec_input_ids = torch.cat([dec_input_ids, prediction], dim=1).to(device)
        dec_attention_mask = torch.tensor([1]*len(dec_input_ids[0])).unsqueeze(0).to(device)
        #print(pred)
    pred_text = fr_tokenizer.decode(dec_input_ids[0], skip_special_tokens=True)
    bleu = None
    bert_score = None
    if compute_metrics:
        bleu = bleu_metric.compute(predictions=[pred_text], references=[[fr_text]])
        bert_score = bert_metric.compute(predictions=[pred_text], references=[[fr_text]], lang='fr')
    return pred_text, bleu, bert_score

In [203]:
print(en_list[120])
print(fr_list[120])

Asian man and blond woman holding hands outdoors, man in background watches.
Un homme asiatique et une femme blond tiennent les mains à l'extérieur, l'homme à l'arrière-plan regarde.


In [214]:
en_text = en_list[120]
fr_text = fr_list[120]
fr_start = '<|endoftext|>'
print(en_text, fr_start)
pred_text, bleu, bert_score = translate(en_text, fr_text, fr_start, en_tokenizer, fr_tokenizer, bro_model, True)

Asian man and blond woman holding hands outdoors, man in background watches. <|endoftext|>


In [215]:
print(f'English text: {en_text}')
print(f"Original Text: {fr_text} \nPredicted Text: {pred_text}\n Bleu Score:\n {bleu}\n BertScore:\n  {bert_score}")

English text: Asian man and blond woman holding hands outdoors, man in background watches.
Original Text: Un homme asiatique et une femme blond tiennent les mains à l'extérieur, l'homme à l'arrière-plan regarde. 
Predicted Text: Un homme asiatique et une femme blond tiennent les mains à l'extérieur, l'homme à l'arrière-plan regarde.
 Bleu Score:
 {'score': 100.00000000000004, 'counts': [18, 17, 16, 15], 'totals': [18, 17, 16, 15], 'precisions': [100.0, 100.0, 100.0, 100.0], 'bp': 1.0, 'sys_len': 18, 'ref_len': 18}
 BertScore:
  {'precision': [1.0], 'recall': [1.0], 'f1': [1.0], 'hashcode': 'bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=4.49.0)'}


In [216]:
en_text = en_list[14500]
fr_text = fr_list[14500]
fr_start = '<|endoftext|>'
print(en_text, fr_start)
pred_text, bleu, bert_score = translate(en_text, fr_text, fr_start, en_tokenizer, fr_tokenizer, bro_model, True)
print(f'English text: {en_text}')
print(f"Original Text: {fr_text} \nPredicted Text: {pred_text}\n Bleu Score:\n {bleu}\n BertScore:\n  {bert_score}")

A woman with a black and white sweater standing on a walkway. <|endoftext|>
English text: A woman with a black and white sweater standing on a walkway.
Original Text: Une femme en chandail noir et blanc se tient sur une passerelle. 
Predicted Text: Une femme en chandail noir et blanc se tient sur une passerelle.
 Bleu Score:
 {'score': 100.00000000000004, 'counts': [13, 12, 11, 10], 'totals': [13, 12, 11, 10], 'precisions': [100.0, 100.0, 100.0, 100.0], 'bp': 1.0, 'sys_len': 13, 'ref_len': 13}
 BertScore:
  {'precision': [1.0], 'recall': [1.0], 'f1': [1.0], 'hashcode': 'bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=4.49.0)'}


In [217]:
en_text = en_list[3500]
fr_text = fr_list[3500]
fr_start = '<|endoftext|>'
print(en_text, fr_start)
pred_text, bleu, bert_score = translate(en_text, fr_text, fr_start, en_tokenizer, fr_tokenizer, bro_model, True)
print(f'English text: {en_text}')
print(f"Original Text: {fr_text} \nPredicted Text: {pred_text}\n Bleu Score:\n {bleu}\n BertScore:\n  {bert_score}")

A child in a martial arts uniform is jumping in the air with his arms and legs spread. <|endoftext|>
English text: A child in a martial arts uniform is jumping in the air with his arms and legs spread.
Original Text: Un enfant en uniforme d'arts martiaux saute dans l'air avec ses bras et ses jambes étalés. 
Predicted Text: Un enfant en uniforme d'arts martiaux saute dans l'air avec ses bras et ses jambes étalés.
 Bleu Score:
 {'score': 100.00000000000004, 'counts': [17, 16, 15, 14], 'totals': [17, 16, 15, 14], 'precisions': [100.0, 100.0, 100.0, 100.0], 'bp': 1.0, 'sys_len': 17, 'ref_len': 17}
 BertScore:
  {'precision': [0.9999999403953552], 'recall': [0.9999999403953552], 'f1': [0.9999999403953552], 'hashcode': 'bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=4.49.0)'}


In [218]:
en_text = "I am a good boy"
fr_start = '<|endoftext|>'
print(en_text, fr_start)
translate(en_text,None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

I am a good boy <|endoftext|>


('Je suis un bon garçon', None, None)

In [220]:
en_text = "You know. Today is a hot day"
fr_start = '<|endoftext|>'
print(en_text, fr_start)
translate(en_text, None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

You know. Today is a hot day <|endoftext|>


('Vous savez que la journée (tension est une journée chaude.', None, None)

In [222]:
en_text = "I am a student"
fr_start = '<|endoftext|>'
print(en_text, fr_start)
translate(en_text, None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

I am a student <|endoftext|>


('Je suis un étudiant', None, None)

In [187]:
en_text = "I saw a beautiful girl"
fr_start = '<|endoftext|>'
#fr_start = 'Je'
print(en_text, fr_start)
translate(en_text, fr_start, en_tokenizer, fr_tokenizer, bro_model)

I saw a beautiful girl <|endoftext|>


"J'ai vu une belle fille"

In [223]:
en_text = "A boy is going to school"
fr_start = '<|endoftext|>'
#fr_start = 'Un'
print(en_text, fr_start)
translate(en_text, None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

A boy is going to school <|endoftext|>


("Un garçon va à l'école", None, None)

In [224]:
en_text = "This is my car"
fr_start = '<|endoftext|>'
#fr_start = 'Un'
print(en_text, fr_start)
translate(en_text, None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

This is my car <|endoftext|>


("C'est ma voiture que ma voiture est ma voiture.", None, None)

In [225]:
en_text = "It is a beautiful day and I am loving it"
fr_start = '<|endoftext|>'
#fr_start = 'Un'
print(en_text, fr_start)
translate(en_text, None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

It is a beautiful day and I am loving it <|endoftext|>


("C'est une belle journée et je l'ai aimée.", None, None)

In [226]:
en_text = "The colour of house is yellow"
fr_start = '<|endoftext|>'
#fr_start = 'Un'
print(en_text, fr_start)
translate(en_text, None, fr_start, en_tokenizer, fr_tokenizer, bro_model)

The colour of house is yellow <|endoftext|>


('Le personnel de la maison est jaune', None, None)

In [189]:
## So we our translation system is working good. The first character missing issue has been resolved and it is looking pretty good

In [230]:
##All done. Perform french to english translation using sagemaker script mode . Implement beam search